In [21]:
import torch
import torch.nn as nn
import numpy as np
import re
from collections import Counter

In [22]:

# Read file
with open("shakespeare.txt", 'r') as f:
    text = f.read()


print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [23]:
text = text.lower()
text = re.sub(r'[^a-z\s]', '', text)

words = text.split()

print("Total words:", len(words))

Total words: 202619


In [24]:
words = words[:80000]
print("Reduced words:", len(words))

Reduced words: 80000


In [25]:
vocab = sorted(set(words))

word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}

vocab_size = len(vocab)
print("Vocab size:", vocab_size)

Vocab size: 7565


In [26]:
seq_length = 5
X = []
y = []

for i in range(len(words) - seq_length):
    seq = words[i:i+seq_length]
    target = words[i+seq_length]
    
    X.append([word_to_idx[w] for w in seq])
    y.append(word_to_idx[target])

X = torch.tensor(X)
y = torch.tensor(y)

print("Shape:", X.shape)

Shape: torch.Size([79995, 5])


In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 100)
        self.lstm = nn.LSTM(100, 512, batch_first=True)
        self.fc = nn.Linear(512, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)

model = LSTMModel(vocab_size)

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

model.to(device)
X = X.to(device)
y = y.to(device)

Using device: cuda


In [29]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [30]:
epochs = 30
batch_size = 128

for epoch in range(epochs):
    total_loss = 0
    
    for i in range(0, len(X), batch_size):
        xb = X[i:i+batch_size]
        yb = y[i:i+batch_size]
        
        optimizer.zero_grad()
        output = model(xb)
        loss = criterion(output, yb)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (128x512 and 256x7565)

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'word_to_idx': word_to_idx,
    'idx_to_word': idx_to_word
}, "shakespeare_lstm.pt")

print("Model saved!")

In [ ]:
def predict_next(text_seq):
    model.eval()
    
    words_input = text_seq.lower().split()[-5:]
    seq = [word_to_idx.get(w, 0) for w in words_input]
    seq = torch.tensor(seq).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(seq)
        pred = torch.argmax(output).item()
    
    return idx_to_word[pred]

print(predict_next("to be or not to"))